# Лекція 11 — pandas глибоке занурення на Stack Overflow Developer Survey 2025

**Курс:** Applied Software Development (Python) 2026

---

## Передумови (Prerequisites)

Ця лекція **повністю самодостатня**: жодної залежності від проєктного коду з лекцій 6–10. Нам не потрібні ані вебфреймворк, ані база даних, ані контейнери — лише Python, Jupyter та один CSV.

Достатньо знань з **Лекцій 1–5**:

- типи даних, змінні, f-strings (Л1)
- колекції: `list`, `dict`, `tuple`, `set` (Л2–Л3)
- генератори списків / словників (Л3)
- функції, `*args`/`**kwargs`, lambda (Л3–Л4)
- базове файлове введення-виведення (Л5)

Плюс встановлений **Jupyter** (notebook, lab або VS Code) і можливість запустити `pip install`.

---

## Датасет, на якому все побудовано

Один CSV — **Stack Overflow Annual Developer Survey 2025** (~49 000 респондентів, 177 країн). Ми завантажимо його з офіційного джерела під ліцензією **ODbL**, почистимо, поріжемо, згрупуємо, з'єднаємо — і побачимо, як виглядає світ розробників у цифрах.

> **Важливо:** CSV-файл **не** вкомітчений у репозиторій (він завеликий). Ви завантажуєте його самі — див. `README.md` у теці цієї лекції.


## Цілі заняття

Після цієї лекції ви зможете:

1. Пояснити, **коли pandas — правильний інструмент**, а коли варто дивитися в бік DuckDB чи Polars.
2. Розрізнити `Series` та `DataFrame` і будувати їх з чистого Python (без зовнішніх файлів).
3. Завантажити справжній CSV з `pd.read_csv` та вивчити його: типи, пропущені значення, примусове приведення типів (`pd.to_numeric(errors="coerce")`).
4. Впевнено користуватися `.loc`, `.iloc`, булевими масками, `.query()`, `.isin()`, `.explode()`, `groupby().agg()`, `merge()`, `pivot_table` та `crosstab`.
5. Застосовувати інтермедіат-патерни: method chaining (`.pipe`, `.assign`), `.apply` / `.map`, та `Categorical` dtype з реальним порівнянням використання пам'яті.

## 1. Для чого pandas — і для чого ні

**pandas** — це Python-бібліотека для роботи з табличними даними. Синтаксично схожа на Excel, але в сто разів швидша і в тисячу разів програмованіша.

### Сильні сторони

- **Векторизовані операції над колонками (vectorization).** `df["a"] + df["b"]` — один виклик у C під капотом, а не Python-цикл. Це зазвичай у 10–1000 разів швидше, ніж `for row in df: ...`.
- **Багатий API для злиття, групування, перетворень.** `groupby`, `merge`, `pivot_table`, `rolling`, `resample` — все вбудовано і добре документовано.
- **Екосистема.** pandas легко інтегрується з matplotlib, seaborn, scikit-learn, Jupyter, Parquet, Excel, різними SQL-базами — одним словом, з усім.

### Слабкі сторони

- **Все в пам'яті, один процес.** Якщо датасет не вміщується в RAM — pandas зламається. Жодних out-of-core чи паралельних обчислень (крім зовнішніх костилів).
- **"Магія" індексів.** `.loc` vs `.iloc`, `SettingWithCopyWarning`, view vs copy — pandas має свою філософію, яка не віразу очевидна.

**Правило великого пальця:** якщо ваш датасет вміщується в пам'ять (≤ кілька GB) і ви робите з ним переважно **агрегації, перетворення та поєднання** — pandas ідеальний. Якщо ні — дивіться в бік DuckDB чи Polars (Секція 16).


## 2. Series vs DataFrame

Перш ніж завантажити справжні дані, збудуємо обидва основні об'єкти pandas **з чистого Python**, щоб побачити їхню природу.

**Ментальна модель:**

- `Series` — одновимірний масив значень з **індексом** (ярликами).
- `DataFrame` — словник `Series`-ів, які **ділять між собою один Index**.

Тобто `DataFrame` — це таблиця, яку можна подивитися як "набір колонок" (кожна колонка — `Series`).


In [16]:
import pandas as pd

# Series з  Python-списку — pandas сам створить цілочисловий Index 0..N-1
temps = pd.Series([18.5, 21.0, 19.8, 22.3], name="temperature_c")
print(temps)
print("---")
print(f"dtype: {temps.dtype}")
print(f"name:  {temps.name}")
print(f"index: {list(temps.index)}")

0    18.5
1    21.0
2    19.8
3    22.3
Name: temperature_c, dtype: float64
---
dtype: float64
name:  temperature_c
index: [0, 1, 2, 3]


In [17]:
# Series з ярликами (індексами): явно задаємо Index
rating = pd.Series(
    data=[4.7, 4.5, 4.9, 4.2],
    index=["Python", "JavaScript", "Rust", "PHP"],
    name="developer_love",
)
print(rating)
print("---")
# Доступ за ярликом, не за позицією
print("Rust:", rating["Rust"])

Python        4.7
JavaScript    4.5
Rust          4.9
PHP           4.2
Name: developer_love, dtype: float64
---
Rust: 4.9


In [18]:
# DataFrame = dict of Series з одним спільним Index
devs = pd.DataFrame(
    {
        "country": ["Ukraine", "USA", "Poland", "Germany"],
        "years_pro": [5, 9, 4, 12],
        "uses_python": [True, True, False, True],
    }
)
print(devs)
print("---")
# Кожна колонка — Series; індекс 0..3 спільний для всіх колонок
print(type(devs["country"]))
print(devs["country"])

   country  years_pro  uses_python
0  Ukraine          5         True
1      USA          9         True
2   Poland          4        False
3  Germany         12         True
---
<class 'pandas.core.series.Series'>
0    Ukraine
1        USA
2     Poland
3    Germany
Name: country, dtype: object


**Ключова ідея:** коли ми потім напишемо `df["Country"]` над справжнім CSV — повернеться саме `Series`. Коли ми напишемо `df[["Country", "DevType"]]` (зверніть увагу, подвійні дужки) — повернеться `DataFrame` з підмножиною колонок.

## 3. Завантаження даних: pd.read_csv на Stack Overflow Survey 2025

Тепер — до справжнього датасету. Pinуємо версію зверху, щоб лекція була детерміністичною:


In [19]:
from pathlib import Path

# Пінимо рік Survey, щоб лекція була детерміністичною.
# Схеми колонок дрейфують рік за роком — якщо перейдете на новіший Survey,
# перевірте df.columns.tolist() і оновіть список нижче.
SURVEY_YEAR = 2025
SURVEY_PATH = Path("data") / "survey_results_public.csv"

print(f"Survey рік: {SURVEY_YEAR}")
print(f"Очікуваний шлях: {SURVEY_PATH.resolve()}")
print(f"Файл існує: {SURVEY_PATH.exists()}")

Survey рік: 2025
Очікуваний шлях: D:\applied_software_development_python_2026\lectures\11-pandas-analytics\data\survey_results_public.csv
Файл існує: True


### Якщо CSV відсутній

Якщо `data/survey_results_public.csv` ще не на місці — ноутбук зупиниться на наступній клітинці з ясним повідомленням. Щоб це виправити:

1. Відкрийте <https://survey.stackoverflow.co/> і знайдіть посилання **"Download the full data set (CSV)"**.
2. Розпакуйте ZIP-архів.
3. Покладіть `survey_results_public.csv` за шляхом `lectures/11-pandas-analytics/data/survey_results_public.csv`.
4. Перезапустіть ноутбук з початку.

> **Без реального CSV лекція не запрацює**

In [20]:
# Колонки, які нам знадобляться протягом лекції.
# Повний CSV 2025 Survey має 172 колонки; завантажимо лише потрібні для швидкості.
USECOLS = [
    "ResponseId",
    "MainBranch",
    "Country",
    "EdLevel",
    "YearsCode",        
    "WorkExp",           
    "DevType",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "DatabaseHaveWorkedWith",
    "ConvertedCompYearly",
    "RemoteWork",
]

if not SURVEY_PATH.exists():
    raise FileNotFoundError(
        f"Не знайдено {SURVEY_PATH.resolve()}.\n"
        f"Завантажте ZIP з https://survey.stackoverflow.co/, розпакуйте, "
        f"і покладіть survey_results_public.csv у теку lectures/11-pandas-analytics/data/. "
        f"Деталі — у README.md."
    )

df = pd.read_csv(
    SURVEY_PATH,
    usecols=USECOLS,
    na_values=["NA", "N/A", ""],
    low_memory=False,
)
print(f"✅ Завантажено Survey {SURVEY_YEAR}: {len(df):,} рядків × {df.shape[1]} колонок")


✅ Завантажено Survey 2025: 49,191 рядків × 12 колонок


In [21]:
# Базова "розвідка" датасету
df.head()

,ResponseId,MainBranch,EdLevel,WorkExp,YearsCode,DevType,RemoteWork,Country,LanguageHaveWorkedWith,LanguageWantToWorkWith,DatabaseHaveWorkedWith,ConvertedCompYearly
0,1,I am a developer by profession,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",8.0,14.0,"Developer, mobile",Remote,Ukraine,Bash/Shell (all shells);Dart;SQL,Dart,Cloud Firestore;PostgreSQL,61256.0
1,2,I am a developer by profession,"Associate degree (A.A., A.S., etc.)",2.0,10.0,"Developer, back-end","Hybrid (some in-person, leans heavy to flexibi...",Netherlands,Java,Java;Python;Swift,Dynamodb;MongoDB,104413.0
2,3,I am a developer by profession,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",10.0,12.0,"Developer, front-end",NaN,Ukraine,Dart;HTML/CSS;JavaScript;TypeScript,Dart;HTML/CSS;JavaScript;TypeScript,MongoDB;MySQL;PostgreSQL,53061.0
3,4,I am a developer by profession,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",4.0,5.0,"Developer, back-end",Remote,Ukraine,Java;Kotlin;SQL,Java;Kotlin,NaN,36197.0
4,5,I am a developer by profession,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",21.0,22.0,Engineering manager,NaN,Ukraine,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...,C#;Java;JavaScript;Python;SQL;TypeScript,Elasticsearch;Microsoft SQL Server;MySQL;Oracl...,60000.0


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ResponseId              49191 non-null  int64  
 1   MainBranch              49191 non-null  object 
 2   EdLevel                 48149 non-null  object 
 3   WorkExp                 42893 non-null  float64
 4   YearsCode               43042 non-null  float64
 5   DevType                 43680 non-null  object 
 6   RemoteWork              33780 non-null  object 
 7   Country                 35437 non-null  object 
 8   LanguageHaveWorkedWith  31671 non-null  object 
 9   LanguageWantToWorkWith  27079 non-null  object 
 10  DatabaseHaveWorkedWith  25550 non-null  object 
 11  ConvertedCompYearly     23947 non-null  float64
dtypes: float64(3), int64(1), object(8)
memory usage: 4.5+ MB


In [24]:
df.describe(include="all").T.head(15)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ResponseId,49191.0,NaN,NaN,NaN,24596.0,14200.362883,1.0,12298.5,24596.0,36893.5,49191.0
MainBranch,49191,6,I am a developer by profession,37467,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EdLevel,48149,8,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",20278,NaN,NaN,NaN,NaN,NaN,NaN,NaN
WorkExp,42893.0,NaN,NaN,NaN,13.367403,10.800117,1.0,5.0,10.0,20.0,100.0
YearsCode,43042.0,NaN,NaN,NaN,16.570861,11.78761,1.0,8.0,14.0,24.0,100.0
DevType,43680,32,"Developer, full-stack",12351,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RemoteWork,33780,5,Remote,10931,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,35437,177,United States of America,7233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LanguageHaveWorkedWith,31671,15478,HTML/CSS;JavaScript;TypeScript,370,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LanguageWantToWorkWith,27079,13740,Rust,554,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# Розмір, колонки, типи
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("---")
print(df.dtypes)

Shape: (49191, 12)
Columns: ['ResponseId', 'MainBranch', 'EdLevel', 'WorkExp', 'YearsCode', 'DevType', 'RemoteWork', 'Country', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'ConvertedCompYearly']
---
ResponseId                  int64
MainBranch                 object
EdLevel                    object
WorkExp                   float64
YearsCode                 float64
DevType                    object
RemoteWork                 object
Country                    object
LanguageHaveWorkedWith     object
LanguageWantToWorkWith     object
DatabaseHaveWorkedWith     object
ConvertedCompYearly       float64
dtype: object


## 4. Чистка даних: типи та пропущені значення

![pandas: data science](https://media.licdn.com/dms/image/v2/C4E22AQG6xEySqiBIgw/feedshare-shrink_800/feedshare-shrink_800/0/1657120777466?e=2147483647&v=beta&t=c8xIKXAfhftvcczvrSSfURSEhIlFL1axkNLRgx9SaXo)

У справжніх CSV завжди дві проблеми:

1. **Пропущені значення (NaN).** Респонденти не зобов'язані відповідати на кожне питання.
2. **Неправильні типи.** Колонка, яку ми очікуємо числовою, інколи приїжджає як `object` (рядок), бо в ній сидять "граничні маркери" типу `"Less than 1 year"`, `"More than 50 years"` або просто сторонній текст.

> **Увага:** 2025 Survey **уже добре причесаний** — `YearsCode` і `WorkExp` приїжджають як `float64`. У попередніх роках Survey (2023, 2024) ті самі колонки були `object` з рядковими граничними маркерами. Техніку `pd.to_numeric(errors="coerce")` ви все одно мусите знати — ви зустрінете її в інших CSV щотижня.

> **Примітка про дати:** у 2025 Survey нема колонки з датою/часом відповіді, тож `pd.to_datetime` тут не показуємо — повернемося до нього в лекції, де є справжні часові ряди.

### 4.1. Скільки NaN у кожній колонці?


In [27]:
df.isna().sum().sort_values(ascending=False)

ConvertedCompYearly       25244
DatabaseHaveWorkedWith    23641
LanguageWantToWorkWith    22112
LanguageHaveWorkedWith    17520
RemoteWork                15411
Country                   13754
WorkExp                    6298
YearsCode                  6149
DevType                    5511
EdLevel                    1042
ResponseId                    0
MainBranch                    0
dtype: int64

### 4.2. `pd.to_numeric(errors="coerce")` — на прикладі "брудної" Series

Показуємо техніку на невеличкій **синтетичній** Series, щоб ви її впізнали, коли зустрінете в іншому CSV. У нашому 2025 Survey ця техніка — no-op (бо числа вже є), але запустити її безпечно в будь-якому разі.


In [28]:
# Демонстраційна Series із рядковими "граничними" маркерами
# (саме такий вигляд мав YearsCodePro в Survey 2023/2024)
messy = pd.Series(["1", "5", "Less than 1 year", "20", "More than 50 years",
                   None, "не знаю", "10"])
print("ДО: dtype =", messy.dtype)
print(messy.tolist())
print()

# Крок 1: replace — підміняємо відомі рядкові маркери на числа
replacements = {"Less than 1 year": 0.5, "More than 50 years": 51.0}
# Крок 2: to_numeric із errors="coerce" — решту нерозпізнаного перетворює на NaN
cleaned = pd.to_numeric(messy.replace(replacements), errors="coerce")

print("ПІСЛЯ: dtype =", cleaned.dtype)
print(cleaned.tolist())

ДО: dtype = object
['1', '5', 'Less than 1 year', '20', 'More than 50 years', None, 'не знаю', '10']

ПІСЛЯ: dtype = float64
[1.0, 5.0, 0.5, 20.0, 51.0, nan, nan, 10.0]


Принципово: `errors="coerce"` — **не** викидає виняток на незрозумілому значенні, а перетворює його на `NaN`. Це дає вам шанс побачити проблемні рядки (`cleaned.isna()`) і вирішити, що з ними робити, замість того щоб зупинити весь pipeline.

### 4.3. Перевіримо на реальних даних 2025 Survey


In [29]:
# У 2025 ці колонки вже числові — виклик безпечний (no-op для вже-числових)
df["YearsCode"] = pd.to_numeric(df["YearsCode"], errors="coerce")
df["WorkExp"] = pd.to_numeric(df["WorkExp"], errors="coerce")

print(df[["YearsCode", "WorkExp"]].dtypes)
df[["YearsCode", "WorkExp"]].describe()

YearsCode    float64
WorkExp      float64
dtype: object


,YearsCode,WorkExp
count,43042.000000,42893.000000
mean,16.570861,13.367403
std,11.787610,10.800117
min,1.000000,1.000000
25%,8.000000,5.000000
50%,14.000000,10.000000
75%,24.000000,20.000000
max,100.000000,100.000000


### 4.3. Що робити з NaN: dropna vs fillna

Два підходи, і вибір залежить від питання:

- `dropna(subset=[...])` — "мене цікавлять лише респонденти, які відповіли на конкретне питання".
- `fillna(value)` — "відсутня відповідь насправді означає щось відоме, і я хочу це кодувати".


In [30]:
# Drop будь-які рядки, де нема WorkExp — бо для наших агрегацій потрібен стаж
n_before = len(df)
df_with_pro = df.dropna(subset=["WorkExp"])
n_after = len(df_with_pro)
print(f"dropna(subset=['WorkExp']): {n_before:,} → {n_after:,} рядків "
      f"(викинули {n_before - n_after:,})")

dropna(subset=['WorkExp']): 49,191 → 42,893 рядків (викинули 6,298)


In [31]:
# Fillna: для зручного звіту замінимо відсутній DevType на явну мітку
dev_type_filled = df["DevType"].fillna("— не вказано —")
dev_type_filled.value_counts().head(10)

DevType
Developer, full-stack                            12351
Developer, back-end                               6453
— не вказано —                                    5511
Student                                           3008
Architect, software or solutions                  2684
Developer, front-end                              1974
Developer, desktop or enterprise applications     1919
Other (please specify):                           1825
Developer, mobile                                 1391
Developer, embedded applications or devices       1274
Name: count, dtype: int64

## 5. Індексація та вибірка

Чотири інструменти, які закривають 95% випадків:

- **`.loc[rows, cols]`** — за ярликами (label-based).
- **`.iloc[rows, cols]`** — за позиціями (positional).
- **Булеві маски** — `df[df["Country"] == "Ukraine"]`.
- **`.query("Country == 'Ukraine'")`** — читабельна версія булевої маски, добре комбінується.

### 5.1. .loc і .iloc


In [32]:
# .loc — за ярликами (індекс + імена колонок)
# Перші 3 рядки (індекси 0, 1, 2) і дві конкретні колонки
df.loc[0:2, ["Country", "WorkExp"]]

,Country,WorkExp
0,Ukraine,8.0
1,Netherlands,2.0
2,Ukraine,10.0


In [33]:
# .iloc — за позиціями (0-based, як Python-списки)
# Перші 3 рядки і колонки 0, 2, 4
df.iloc[0:3, [0, 2, 4]]

,ResponseId,EdLevel,YearsCode
0,1,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",14.0
1,2,"Associate degree (A.A., A.S., etc.)",10.0
2,3,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",12.0


### 5.2. Булеві маски + пастка з дужками

Комбінувати маски потрібно через `&`, `|`, `~` (не `and`/`or`/`not`!) — **і кожну маску загортати в дужки**. Це класична помилка-новачка.


In [34]:
# ПРАВИЛЬНО: дужки навколо кожної маски
mask = (df["WorkExp"] >= 5) & (df["ConvertedCompYearly"].notna())
n = mask.sum()
print(f"Респондентів з ≥5 років стажу і вказаною компенсацією: {n:,}")

Респондентів з ≥5 років стажу і вказаною компенсацією: 19,123


In [35]:
# .isin() — перевірка належності до множини
top5_countries = ["Ukraine", "United States of America", "Germany", "India", "United Kingdom"]
df_top5 = df[df["Country"].isin(top5_countries)]
df_top5["Country"].value_counts()

Country
United States of America    7233
Germany                     3025
India                       2547
Ukraine                      964
Name: count, dtype: int64

## 6. Багатозначні стовпці та `.explode()`

Survey любить зберігати кілька значень в одній клітинці через крапку з комою:

```text
LanguageHaveWorkedWith:  "Python;JavaScript;Go"
```

Щоб порахувати, скільки респондентів використовують кожну мову, треба **розгорнути** це в довгу форму — один рядок на одну мову.

![explode-schematic](assets/diagrams/explode-schematic.png)

Два кроки:

1. `str.split(";")` — перетворює рядок на список.
2. `.explode(col)` — розгортає кожен елемент списку в окремий рядок.


In [36]:
# Будуємо "довгу" таблицю: один рядок = один респондент × одна мова
languages_long = (
    df.assign(LanguageList=df["LanguageHaveWorkedWith"].str.split(";"))
      .explode("LanguageList")
      .rename(columns={"LanguageList": "Language"})
      [["ResponseId", "Country", "Language"]]
      .dropna(subset=["Language"])
)

print(f"df: {len(df):,} рядків")
print(f"languages_long: {len(languages_long):,} рядків (один респондент → кілька)")
languages_long.head(10)

df: 49,191 рядків
languages_long: 194,584 рядків (один респондент → кілька)


,ResponseId,Country,Language
0,1,Ukraine,Bash/Shell (all shells)
0,1,Ukraine,Dart
0,1,Ukraine,SQL
1,2,Netherlands,Java
2,3,Ukraine,Dart
2,3,Ukraine,HTML/CSS
2,3,Ukraine,JavaScript
2,3,Ukraine,TypeScript
3,4,Ukraine,Java
3,4,Ukraine,Kotlin


In [37]:
# Топ-15 мов серед усіх респондентів
top_langs = languages_long["Language"].value_counts().head(15)
top_langs

Language
JavaScript                 21005
HTML/CSS                   19698
SQL                        18633
Python                     18410
Bash/Shell (all shells)    15503
TypeScript                 13859
Java                        9358
C#                          8852
C++                         7485
PowerShell                  7371
C                           6987
PHP                         5994
Go                          5219
Rust                        4724
Kotlin                      3420
Name: count, dtype: int64

## 7. Groupby та агрегації — де ми в цифрах?

`groupby` — найчастіше вживаний інструмент pandas. Патерн завжди один:

```text
df.groupby(<ключ>)[<колонки>].<агрегація>()
```

**У цьому розділі ми використовуємо Україну як наскрізний якір** — побачимо, як наша країна виглядає на глобальному тлі.

### 7.1. Медіанна компенсація — Україна vs Global


In [38]:
# Цікавий патерн: згрупувати за булевим, що перетворюється на легку мітку
ua_mask = df["Country"] == "Ukraine"
ua_vs_global = (
    df.groupby(ua_mask.map({True: "Україна", False: "Глобально"}))
      ["WorkExp"]
      .median()
)
ua_vs_global

Country
Глобально    10.0
Україна       6.0
Name: WorkExp, dtype: float64

### 7.2. Один ключ: медіанна компенсація за країною (топ-20)

In [55]:
median_comp_by_country = (
    df.dropna(subset=["ConvertedCompYearly"])
      .groupby("Country")["ConvertedCompYearly"]
      .median()
      .sort_values(ascending=False)
      .head(20)
)
median_comp_by_country

Country
Oman                                                    390135.0
North Korea                                             233924.5
Andorra                                                 200125.5
United States of America                                150000.0
Switzerland                                             142592.0
Israel                                                  141188.0
Nomadic                                                 135157.5
Mauritania                                              121570.0
Iceland                                                 117637.0
Ireland                                                 116015.0
Belize                                                  102121.0
Denmark                                                  98678.0
Luxembourg                                               98612.0
Australia                                                97514.0
Jamaica                                                  96796.5
United Kingdom of

## 8. Merge / join — зливаємо дві таблиці

Часто треба приєднати результат групування назад до рядків-респондентів ("для кожного респондента покажи, яка середня компенсація у його країні"). Саме для цього `merge`.

Ключові параметри:

- `how=` — `"inner"` (за замовчуванням), `"left"`, `"right"`, `"outer"`.
- `on=` / `left_on=` / `right_on=` — за якою колонкою з'єднуємо.
- `validate=` — перевірка форми з'єднання. Врятує вас від випадкового `many-to-many`, який несподівано перетворює 50 000 рядків на 500 000.

### 8.1. Inner join: мови → статистика → назад до респондентів


In [66]:
# Крок 1 — побудувати per-language агрегати
lang_stats = (
    languages_long.groupby("Language")
                  .agg(n_users=("ResponseId", "nunique"))
                  .reset_index()
                  .sort_values("n_users", ascending=False)
)
lang_stats.head(10)

,Language,n_users
19,JavaScript,21005
17,HTML/CSS,19698
35,SQL,18633
31,Python,18410
2,Bash/Shell (all shells),15503
38,TypeScript,13859
18,Java,9358
4,C#,8852
5,C++,7485
29,PowerShell,7371


In [71]:
# Крок 2 — приєднати ці агрегати до кожного рядка languages_long
# Inner merge: тільки збіги (що у нас і є, бо lang_stats побудовано з languages_long)
enriched = languages_long.merge(
    lang_stats,
    on="Language",
    how="inner",
    validate="many_to_one",  # ← КЛЮЧОВА перевірка: одна мова → одна статистика
)
print(f"До merge: {len(languages_long):,} рядків")
print(f"Після merge: {len(enriched):,} рядків (те саме число — bo many_to_one)")
enriched.head(5)

До merge: 194,584 рядків
Після merge: 194,584 рядків (те саме число — bo many_to_one)


,ResponseId,Country,Language,n_users
0,1,Ukraine,Bash/Shell (all shells),15503
1,1,Ukraine,Dart,1885
2,1,Ukraine,SQL,18633
3,2,Netherlands,Java,9358
4,3,Ukraine,Dart,1885


### 8.2. Left join: зберегти всіх, навіть тих, кому пари нема

In [68]:
# Невелика допоміжна таблиця: "рекомендації" для мов (синтетичне, для демо)
recs = pd.DataFrame({
    "Language": ["Python", "JavaScript", "Rust"],
    "Recommendation": ["✓ solid", "✓ essential", "★ growing"],
})

with_recs = lang_stats.head(10).merge(
    recs, on="Language", how="left", validate="one_to_one",
)
with_recs

,Language,n_users,Recommendation
0,JavaScript,21005,✓ essential
1,HTML/CSS,19698,NaN
2,SQL,18633,NaN
3,Python,18410,✓ solid
4,Bash/Shell (all shells),15503,NaN
5,TypeScript,13859,NaN
6,Java,9358,NaN
7,C#,8852,NaN
8,C++,7485,NaN
9,PowerShell,7371,NaN


## 9. Pivot tables

`pivot_table` — "що показати в клітинках таблиці, де рядки — одне, а колонки — інше?".


In [69]:
# Pivot: медіанна компенсація, рядки = Country, колонки = RemoteWork
pivot = df.pivot_table(
    index="Country",
    columns="RemoteWork",
    values="ConvertedCompYearly",
    aggfunc="median",
    margins=True,          # додає рядок/колонку "All"
    margins_name="Усі",
)
# Обмежимо вивід кількома країнами, щоб було компактно
shown = ["Ukraine", "United States of America", "Germany", "India", "Усі"]
pivot.loc[[c for c in shown if c in pivot.index]]

RemoteWork,"Hybrid (some in-person, leans heavy to flexibility)","Hybrid (some remote, leans heavy to in-person)",In-person,Remote,"Your choice (very flexible, you can come in when you want or just as needed)",Усі
Country,,,,,,
Ukraine,24483.0,13605.5,9107.0,28000.0,11991.5,24000.0
United States of America,145000.0,140000.0,110000.0,165000.0,150000.0,150000.0
Germany,83531.0,75410.0,66824.0,92812.0,83531.0,81210.0
India,24410.0,23248.0,9764.0,20923.0,20923.0,17436.0
Усі,80966.0,72958.0,46406.0,88492.0,75410.0,75410.0


In [ ]:
# Crosstab з normalize="index": "яка частка мов 'хочу вивчити' у кожному DevType?"
# Спершу збудуємо довгу таблицю для "хочу вивчити"
want_long = (
    df.assign(WantList=df["LanguageWantToWorkWith"].str.split(";"))
      .explode("WantList")
      .rename(columns={"WantList": "WantLanguage"})
      .dropna(subset=["WantLanguage", "DevType"])
)

# Фокус: частка тих, хто хоче Rust, за DevType
want_long["wants_rust"] = want_long["WantLanguage"] == "Rust"
rust_share = pd.crosstab(
    want_long["DevType"],
    want_long["wants_rust"],
    normalize="index",
).rename(columns={True: "Хоче Rust", False: "Не хоче Rust"})
rust_share.sort_values("Хоче Rust", ascending=False).head(10)

## 10. Сортування, ранжування, top-N

Три патерни:

- `.sort_values(by=[...], ascending=[...])` — класичне сортування.
- `.nlargest(N, by=...)` / `.nsmallest(N, by=...)` — швидший і чистіший спосіб взяти top-N, ніж `sort + head`.
- `.rank()` — коли треба впорядкувати та зберегти номер позиції (а не просто відсіяти).


In [72]:
# Топ-10 країн за кількістю респондентів
country_counts = df["Country"].value_counts()
country_counts.nlargest(10)

Country
United States of America                                7233
Germany                                                 3025
India                                                   2547
United Kingdom of Great Britain and Northern Ireland    2042
France                                                  1409
Canada                                                  1305
Ukraine                                                  964
Poland                                                   888
Netherlands                                              867
Italy                                                    835
Name: count, dtype: int64

In [73]:
# Те саме, але через sort_values — еквівалентно, але nlargest зазвичай швидший
country_counts.sort_values(ascending=False).head(10)

Country
United States of America                                7233
Germany                                                 3025
India                                                   2547
United Kingdom of Great Britain and Northern Ireland    2042
France                                                  1409
Canada                                                  1305
Ukraine                                                  964
Poland                                                   888
Netherlands                                              867
Italy                                                    835
Name: count, dtype: int64

In [74]:
# Рангування: хто в топ-5, топ-10, топ-20 країн за кількістю респондентів?
ranked = country_counts.rank(method="min", ascending=False).astype(int)
ranked.head(5)

Country
United States of America                                1
Germany                                                 2
India                                                   3
United Kingdom of Great Britain and Northern Ireland    4
France                                                  5
Name: count, dtype: int64

## 11. Method chaining: `.pipe` та `.assign`

Коли pipeline виростає — стиль із проміжними змінними (`df1 = ...; df2 = ...; df3 = ...`) створює смислові "острівці", які легко сплутати. **Method chaining** (ланцюжок викликів) робить pipeline одним логічним блоком.

### 11.1. Stepwise vs chained — той самий pipeline

**Завдання:** з основного DataFrame взяти респондентів-розробників за професією з відомою компенсацією, додати бакет `comp_band` кожної тисячі доларів, вибрати кілька колонок.


In [ ]:
# Варіант А — stepwise, з проміжними змінними
step1 = df[df["MainBranch"] == "I am a developer by profession"]
step2 = step1.dropna(subset=["ConvertedCompYearly"])
step3 = step2.assign(comp_band=(step2["ConvertedCompYearly"] // 25_000) * 25_000)
step4 = step3[["Country", "DevType", "WorkExp", "comp_band"]]

print(step4.shape)
step4.head(5)

In [ ]:
# Варіант Б — chained, один блок. Ідентичний результат.
chained = (
    df.loc[df["MainBranch"] == "I am a developer by profession"]
      .dropna(subset=["ConvertedCompYearly"])
      .assign(comp_band=lambda d: (d["ConvertedCompYearly"] // 25_000) * 25_000)
      [["Country", "DevType", "WorkExp", "comp_band"]]
)

print(chained.shape)
chained.head(5)

**Спостереження.** Обидва варіанти дають той самий результат. Chained-варіант:

- не "протікає" проміжними змінними у namespace;
- читається зверху вниз як послідовність кроків;
- дешевше рефакторити (вставити / прибрати крок = змінити один рядок).

**Коли chaining шкодить:** коли ланцюжок стає > 10 кроків, або коли треба дебажити конкретний проміжний результат. Тоді розбивайте назад на stepwise. Або використайте `.pipe(print)` / `.pipe(lambda d: (display(d.head()), d)[1])` як "дебаг-пробник" усередині ланцюжка.

### 11.2. `.pipe(func)` — коли маєте власну функцію-крок


In [75]:
def drop_outlier_comp(d: pd.DataFrame, lo: float = 1_000, hi: float = 500_000) -> pd.DataFrame:
    """Викинути екстремальні викиди компенсації."""
    return d[(d["ConvertedCompYearly"] >= lo) & (d["ConvertedCompYearly"] <= hi)]


cleaned = (
    df.dropna(subset=["ConvertedCompYearly"])
      .pipe(drop_outlier_comp)
      .assign(comp_band=lambda d: (d["ConvertedCompYearly"] // 25_000) * 25_000)
)
print(f"До: {df.dropna(subset=['ConvertedCompYearly']).shape[0]:,}")
print(f"Після drop_outlier_comp: {cleaned.shape[0]:,}")

До: 23,947
Після drop_outlier_comp: 23,044


## 12. `.apply` та `.map` — власні функції

Коли pandas не має "готової" операції — пишемо свою.

- **`Series.map(dict_or_func)`** — поелементно; найшвидший варіант з тих трьох.
- **`Series.apply(func)`** — поелементно, але приймає функцію; трохи повільніший.
- **`DataFrame.apply(func, axis=0|1)`** — `axis=0` працює по колонках, `axis=1` — по рядках (**найповільніший**).

> **Золоте правило:** перед тим, як тягнутися до `.apply`, запитайте — "чи можна це зробити векторизовано?". Якщо так — робіть векторизовано, буде в 100–1000 разів швидше.


In [76]:
# .map з dict: переклад довгих офіційних міток на коротші
remote_short = df["RemoteWork"].map({
    "Remote": "remote",
    "Hybrid (some remote, some in-person)": "hybrid",
    "In-person": "onsite",
})
remote_short.value_counts(dropna=False)

RemoteWork
NaN       32218
remote    10931
onsite     6042
Name: count, dtype: int64

In [77]:
# .apply з lambda: бакет стажу в людські категорії
def years_bucket(y):
    if pd.isna(y):
        return "невідомо"
    if y < 2:
        return "junior (<2)"
    if y < 5:
        return "mid (2–5)"
    if y < 10:
        return "senior (5–10)"
    return "staff+ (10+)"


df["years_bucket"] = df["WorkExp"].apply(years_bucket)
df["years_bucket"].value_counts()

years_bucket
staff+ (10+)     23921
senior (5–10)     9332
mid (2–5)         7187
невідомо          6298
junior (<2)       2453
Name: count, dtype: int64

### 12.1. Бенчмарк: векторизовано vs `.apply(axis=1)`

Порахуємо "бакет компенсації $25 000" двома способами і заміряємо час.


In [78]:
# Готуємо підмножину, де є компенсація — щоб було що рахувати
comp_df = df.dropna(subset=["ConvertedCompYearly"]).copy()
print(f"Рядків у бенчмарку: {len(comp_df):,}")

Рядків у бенчмарку: 23,947


In [79]:
# Варіант А — векторизовано (одним виразом над цілою колонкою)
%timeit (comp_df["ConvertedCompYearly"] // 25_000) * 25_000

617 μs ± 33.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [80]:
# Варіант Б — через apply(axis=1): проходимося по кожному рядку в Python
%timeit comp_df.apply(lambda row: (row["ConvertedCompYearly"] // 25_000) * 25_000, axis=1)

138 ms ± 14.1 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**Висновок:** векторизований варіант у десятки чи сотні разів швидший. Саме тому `.apply(axis=1)` — останній притулок, а не перший вибір.


In [ ]:
# Вимірюємо використання пам'яті об'єктних колонок
def memory_mb(frame: pd.DataFrame) -> float:
    return frame.memory_usage(deep=True).sum() / 1024 ** 2


before_mb = memory_mb(df)
print(f"ДО оптимізації: {before_mb:.2f} MB")

df_opt = df.copy()
for col in ["Country", "DevType", "MainBranch", "RemoteWork"]:
    df_opt[col] = df_opt[col].astype("category")

after_mb = memory_mb(df_opt)
print(f"ПІСЛЯ конвертації 4 колонок у Categorical: {after_mb:.2f} MB")
print(f"Виграш: {(1 - after_mb / before_mb) * 100:.1f}%")

In [ ]:
# Фільтр "лише Україна" на Categorical-колонці
ua_respondents = df_opt[df_opt["Country"] == "Ukraine"]
print(f"Респондентів з України: {len(ua_respondents):,}")

# Скільки з них має Master's+? Використовуємо той самий позиційний прийом,
# що і в попередній клітинці (cat.codes), щоб обійти особливості порівняння
# Categorical × str у сучасному pandas.
ua_pos = ua_respondents["EdLevel"].cat.codes
ua_master_plus_mask = (ua_pos >= master_pos) & ua_respondents["EdLevel"].notna()
print(f"З них з Master's+ ступенем: {ua_master_plus_mask.sum():,}")

## 13. Практичний блок: три запитання — три пайплайни

Сюди стікається все, що ми вивчили. Кожен пайплайн — ≤ 10 рядків, читабельний зверху вниз.

### 13.1. Топ-10 найпопулярніших мов програмування (глобально)


In [81]:
top10_langs = (
    df.assign(lang_list=df["LanguageHaveWorkedWith"].str.split(";"))
      .explode("lang_list")
      .rename(columns={"lang_list": "Language"})
      .dropna(subset=["Language"])
      ["Language"]
      .value_counts()
      .head(10)
)
top10_langs

Language
JavaScript                 21005
HTML/CSS                   19698
SQL                        18633
Python                     18410
Bash/Shell (all shells)    15503
TypeScript                 13859
Java                        9358
C#                          8852
C++                         7485
PowerShell                  7371
Name: count, dtype: int64

### 13.2. Медіанна компенсація по країнах (топ-20 за кількістю респондентів)

In [82]:
top20_by_resp = df["Country"].value_counts().head(20).index

med_comp = (
    df[df["Country"].isin(top20_by_resp)]
      .dropna(subset=["ConvertedCompYearly"])
      .groupby("Country")
      .agg(
          n=("ResponseId", "size"),
          median_usd=("ConvertedCompYearly", "median"),
      )
      .sort_values("median_usd", ascending=False)
)
med_comp

,n,median_usd
Country,,
United States of America,5243,150000.0
Switzerland,418,142592.0
Denmark,230,98678.0
Australia,569,97514.0
United Kingdom of Great Britain and Northern Ireland,1486,94618.0
Canada,915,87550.0
Germany,2141,81210.0
Netherlands,619,81210.0
Austria,280,78310.0


### 13.3. Хто хоче вивчити Rust — за DevType

In [83]:
rust_by_dev = (
    df.assign(wants=df["LanguageWantToWorkWith"].str.split(";"))
      .explode("wants")
      .dropna(subset=["wants", "DevType"])
      .assign(wants_rust=lambda d: d["wants"] == "Rust")
      .groupby("DevType")
      .agg(
          n_respondents=("ResponseId", "nunique"),
          rust_share=("wants_rust", "mean"),
      )
      .sort_values("rust_share", ascending=False)
      .head(10)
)
rust_by_dev

,n_respondents,rust_share
DevType,,
"Developer, embedded applications or devices",836,0.111084
Cloud infrastructure engineer,250,0.089330
Academic researcher,699,0.087583
AI/ML engineer,385,0.084984
Data engineer,486,0.079199
Cybersecurity or InfoSec professional,238,0.078520
DevOps engineer or professional,696,0.077907
"Developer, back-end",4106,0.077060
Student,1935,0.073383


## 14. Коли pandas не підходить

pandas блискуче працює, доки **весь датасет вміщується в RAM** і **вам достатньо одного процесу**. Коли ці умови ламаються — час шукати інше.

| Інструмент | Масштаб | Стиль API | Коли варто |
|---|---|---|---|
| **pandas** | GB в RAM | Імперативний, рядок за рядком | Аналітика, dev-data, прототип моделей |
| **DuckDB** | TB на диску | SQL (+ Python-bind) | OLAP-запити, файли Parquet/CSV без pre-import у БД |
| **Polars** | GB–TB | Lazy, columnar, Rust-backed | Багатопотокові трансформації, великі pipeline-и |

**Коротко:**

- **DuckDB** — "SQL-рушій, який живе у вашому Python-процесі". Ідеальний, коли маєте купу Parquet/CSV на диску і хочете робити SQL-агрегації без підняття окремої БД.
- **Polars** — "pandas, але багатопотокова, ледача та rust-backed". API схожий на pandas, але швидший на великих обсягах і краще масштабується.

У цій лекції ми **не** встановлюємо жодний з них — це концептуальна карта "куди дивитися далі", а не туторіал. Якщо колись ваш pandas-pipeline почне гальмувати або лізти за межі пам'яті — тепер ви знаєте, куди зазирнути.


## 15. Міні-проєкт: "Developer Survey Insights"

### Частина 1 — Медіанний `WorkExp`: Україна vs Глобально

Порахуйте медіанне значення `WorkExp` для респондентів з України та порівняйте з глобальною медіаною. Виведіть обидва числа та різницю (у роках).

**Підказка:** Секція 7.1 — `groupby` з булевим ключем.

*Спершу спробуйте самі, потім розгорніть розв'язок нижче.*


In [84]:
# ==== РОЗВ'ЯЗОК ЧАСТИНИ 1 ====
ua_median_years_pro = df.loc[df["Country"] == "Ukraine", "WorkExp"].median()
global_median_years_pro = df["WorkExp"].median()
delta_years = ua_median_years_pro - global_median_years_pro

print("Медіана WorkExp:")
print(f"  Україна:    {ua_median_years_pro:.1f} років")
print(f"  Глобальна:  {global_median_years_pro:.1f} років")
print(f"  Різниця:   {delta_years:+.1f} років "
      f"({'Україна нижче' if delta_years < 0 else 'Україна вище'})")

Медіана WorkExp:
  Україна:    6.0 років
  Глобальна:  10.0 років
  Різниця:   -4.0 років (Україна нижче)


### Частина 2 — Топ-10 мов серед тих, хто "вчиться кодити"

Знайдіть 10 найпопулярніших мов програмування серед респондентів з `MainBranch == "I am learning to code"`. Використайте `str.split(";")` + `.explode()`.

**Підказка:** Секція 6 (`.explode`) + Секція 11 (`.nlargest` / `.value_counts`).

*Спершу спробуйте самі, потім розгорніть розв'язок нижче.*


In [85]:
# ==== РОЗВ'ЯЗОК ЧАСТИНИ 2 ====
learners_top_langs = (
    df[df["MainBranch"] == "I am learning to code"]
      .assign(lang_list=lambda d: d["LanguageHaveWorkedWith"].str.split(";"))
      .explode("lang_list")
      .rename(columns={"lang_list": "Language"})
      .dropna(subset=["Language"])
      ["Language"]
      .value_counts()
      .head(10)
)
learners_top_langs

Language
HTML/CSS                   766
Python                     734
JavaScript                 711
SQL                        540
Java                       461
C                          456
C++                        446
Bash/Shell (all shells)    436
TypeScript                 296
C#                         231
Name: count, dtype: int64

## 16. Підсумок

За цю лекцію ми пройшли повний шлях від "чистий Python" до "справжня аналітика на живих даних":

- **Основи:** Series, DataFrame, `pd.read_csv` з `usecols=` та `na_values=`.
- **Чистка:** `pd.to_numeric(errors="coerce")` як безпечний шаблон для приведення типів, `isna`/`fillna`/`dropna` для пропущених значень.
- **Вибірка:** `.loc`, `.iloc`, булеві маски з дужками, `.query()`, `.isin()`.
- **Багатозначні колонки:** `str.split(";")` + `.explode()` — один рядок на одне значення.
- **Агрегації:** `groupby().agg()` з named aggregations
- **Об'єднання:** `merge(..., )`.
- **Переформатування:** `pivot_table` з `margins`.
- **Ранжування:** `.nlargest`, `.sort_values`, `.rank`.
- **Інтермедіат-патерни:** method chaining (`.pipe`, `.assign`), `.apply`/`.map` з бенчмарком проти векторизації.
- **Кордони:** коли pandas ламається і куди дивитися — DuckDB, Polars.

## 18. Джерела

### Дані

- **Stack Overflow Annual Developer Survey 2025** — офіційна сторінка: <https://survey.stackoverflow.co/2025/>
  Ліцензія: **ODbL** (Open Database License). Атрибуцію збережено в цьому ноутбуці.
- Методологія 2025 Survey: <https://survey.stackoverflow.co/2025/methodology/>

### Де повчити Pandas

- Офіційна документація: <https://pandas.pydata.org/docs/>
- "10 minutes to pandas": <https://pandas.pydata.org/docs/user_guide/10min.html>
- pandas Cookbook: <https://pandas.pydata.org/docs/user_guide/cookbook.html>
- курс по Pandas на Kaggle: <https://www.kaggle.com/learn/pandas>
- Wes McKinney, *Python for Data Analysis* (3rd edition, O'Reilly, 2022) — книжка автора pandas, чудове друге джерело.

- Real Python, "pandas GroupBy: Your Guide to Grouping Data in Python": <https://realpython.com/pandas-groupby/>
- KDnuggets, "Pandas: Advanced GroupBy Techniques": <https://www.kdnuggets.com/pandas-advanced-groupby-techniques-for-complex-aggregations>

## 19. Що далі?

**Лекція 12 — NumPy + векторизація + проста ML "з нуля".**

Наступного разу ми:

- зазирнемо під капот pandas — там NumPy-масиви;
- зрозуміємо, чому векторизація швидша за цикли (ви це вже бачили в бенчмарку Секції 12, тепер побачимо **чому**);
- використаємо NumPy щоб побудувати простий класифікатор (логістичну регресію) власними руками, без scikit-learn;
- познайомимося з основними метриками в задачах класифікації (accuracy, precision, recall).

**Лекція 13 — Візуалізація.**

Очищений `df` з Survey 2025, який ми тут зробили, **ще повернеться**: у Л13 ми намалюємо з нього графіки matplotlib / seaborn / plotly. Тож саме тепер добре зберегти ваш pipeline чистим — за два тижні ви його переіспользуєте.